#번역기 만들기

In [20]:
import os
import re
import urllib.request
import zipfile
import sentencepiece as spm
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import random

# GPU 설정 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 중인 PyTorch 버전:", torch.__version__)
print("연결된 디바이스:", device)

# 1. 데이터 다운로드 및 압축 해제
dataset_dir = os.path.expanduser("work/s2s_translation/datasets")
os.makedirs(dataset_dir, exist_ok=True)
zip_path = os.path.join(dataset_dir, "spa-eng.zip")

if not os.path.exists(zip_path):
    print("데이터 다운로드 중...")
    url = "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
    urllib.request.urlretrieve(url, zip_path)
    print("다운로드 완료!")

data_folder = os.path.join(dataset_dir, "spa-eng")
if not os.path.exists(data_folder):
    print("압축 해제 중...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)
    print("압축 해제 완료!")

path_to_file = os.path.join(data_folder, "spa.txt")
print("데이터셋 디렉토리 파일 목록:", os.listdir(dataset_dir))

# 2. 데이터 읽기 및 정제 함수 정의
df = pd.read_csv(path_to_file, sep="\t", names=["eng", "spa"])

def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[^a-zA-Z?.!,]+", " ", sentence)
    sentence = sentence.strip()
    return sentence

# 상위 3만 개 데이터만 사용하여 정제 적용
df = df[:30000]
df["eng"] = df["eng"].apply(preprocess_sentence)
df["spa"] = df["spa"].apply(lambda x: preprocess_sentence(x))

# 3. SentencePiece 학습을 위한 말뭉치(txt) 저장
df["eng"].to_csv("eng_corpus.txt", index=False, header=False, sep="\n", encoding="utf-8")
df["spa"].to_csv("spa_corpus.txt", index=False, header=False, sep="\n", encoding="utf-8")
print("말뭉치 파일 생성 완료: eng_corpus.txt, spa_corpus.txt")

# 4. SentencePiece 토크나이저 모델 학습 및 로드
vocab_size = 3000
pad_id = 0
bos_id = 1
eos_id = 2
unk_id = 3

spm.SentencePieceTrainer.train(
    input="eng_corpus.txt",
    model_prefix="encoder_spm",
    vocab_size=vocab_size,
    pad_id=pad_id,
    bos_id=bos_id,
    eos_id=eos_id,
    unk_id=unk_id
)

spm.SentencePieceTrainer.train(
    input="spa_corpus.txt",
    model_prefix="decoder_spm",
    vocab_size=vocab_size,
    pad_id=pad_id,
    bos_id=bos_id,
    eos_id=eos_id,
    unk_id=unk_id
)

encoder_tokenizer = spm.SentencePieceProcessor()
encoder_tokenizer.load("encoder_spm.model")

decoder_tokenizer = spm.SentencePieceProcessor()
decoder_tokenizer.load("decoder_spm.model")
print("SentencePiece 토크나이저 모델 학습 및 로드 완료!")

# 5

사용 중인 PyTorch 버전: 2.7.1+cu118
연결된 디바이스: cuda
데이터셋 디렉토리 파일 목록: ['spa-eng.zip', 'spa-eng']
말뭉치 파일 생성 완료: eng_corpus.txt, spa_corpus.txt


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: eng_corpus.txt
  input_format: 
  model_prefix: encoder_spm
  model_type: UNIGRAM
  vocab_size: 3000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  d

SentencePiece 토크나이저 모델 학습 및 로드 완료!


input: spa_corpus.txt
  input_format: 
  model_prefix: decoder_spm
  model_type: UNIGRAM
  vocab_size: 3000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_privacy_noise_level: 0
  differential_privacy_clipping_threshold: 0


#모델 학습

In [21]:
import torch.nn as nn
import random

# 하이퍼파라미터 설정 (빠른 학습을 위해 크기 최적화)
VOCAB_SIZE = 3000 
EMB_DIM = 256
HID_DIM = 512
DROPOUT = 0.5

# 1. 인코더 (Encoder)
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hid_dim, dec_hid_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.rnn = nn.GRU(emb_dim, enc_hid_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(enc_hid_dim * 2, dec_hid_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, hidden = self.rnn(embedded)
        # 양방향 GRU의 은닉 상태를 하나로 결합
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))
        return outputs, hidden

# 2. 어텐션 (Attention)
class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()
        self.attn = nn.Linear((enc_hid_dim * 2) + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)
        
    def forward(self, hidden, encoder_outputs):
        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]
        
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return torch.softmax(attention, dim=1)

# 3. 디코더 (Decoder)
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid_dim, dec_hid_dim, dropout, attention):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU((enc_hid_dim * 2) + emb_dim, dec_hid_dim, batch_first=True)
        self.fc_out = nn.Linear((enc_hid_dim * 2) + dec_hid_dim + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, encoder_outputs):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        
        a = self.attention(hidden, encoder_outputs).unsqueeze(1)
        weighted = torch.bmm(a, encoder_outputs)
        
        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))
        
        prediction = self.fc_out(torch.cat((output.squeeze(1), weighted.squeeze(1), embedded.squeeze(1)), dim=1))
        return prediction, hidden.squeeze(0)

# 4. 통합 Seq2Seq 모델
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim
        
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        encoder_outputs, hidden = self.encoder(src)
        
        input = trg[:, 0] # 첫 입력: <bos>
        
        for t in range(1, trg_len):
            output, hidden = self.decoder(input, hidden, encoder_outputs)
            outputs[:, t, :] = output
            
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1) 
            input = trg[:, t] if teacher_force else top1
            
        return outputs

# 인스턴스화 및 디바이스 할당
enc = Encoder(VOCAB_SIZE, EMB_DIM, HID_DIM, HID_DIM, DROPOUT)
attn = Attention(HID_DIM, HID_DIM)
dec = Decoder(VOCAB_SIZE, EMB_DIM, HID_DIM, HID_DIM, DROPOUT, attn)

model = Seq2Seq(enc, dec, device).to(device)

print("✅ 모델 설계 및 GPU 메모리 할당 완료!")

✅ 모델 설계 및 GPU 메모리 할당 완료!


#모델 데이터 불러오고 학습

In [22]:
import os, re, random
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import sentencepiece as spm
from tqdm import tqdm

# ==========================================
# 1. 환경 설정 및 기존 데이터/토크나이저 로드
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset_dir = os.path.expanduser("work/s2s_translation/datasets")
path_to_file = os.path.join(dataset_dir, "spa-eng/spa.txt")

# 앞서 다운받은 파일 읽기
df = pd.read_csv(path_to_file, sep="\t", names=["eng", "spa"])
df = df[:30000]

def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[^a-zA-Z?.!,]+", " ", sentence)
    return sentence.strip()

df["eng"] = df["eng"].apply(preprocess_sentence)
df["spa"] = df["spa"].apply(preprocess_sentence)

# 아까 만들어둔 SentencePiece 모델 바로 불러오기 (시간 단축!)
encoder_tokenizer = spm.SentencePieceProcessor()
encoder_tokenizer.load("encoder_spm.model")
decoder_tokenizer = spm.SentencePieceProcessor()
decoder_tokenizer.load("decoder_spm.model")

# ==========================================
# 2. 데이터셋 및 DataLoader 구축
# ==========================================
class TranslationDataset(Dataset):
    def __init__(self, data, encoder_tokenizer, decoder_tokenizer, max_len):
        self.data = data
        self.enc_tok = encoder_tokenizer
        self.dec_tok = decoder_tokenizer
        self.max_len = max_len
        self.pad_id = 0; self.bos_id = 1; self.eos_id = 2
        
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        src_ids = self.enc_tok.encode(self.data.iloc[idx]['eng'])[:self.max_len]
        trg_ids = self.dec_tok.encode(self.data.iloc[idx]['spa'])
        
        trg_input = [self.bos_id] + trg_ids[:self.max_len - 2] + [self.eos_id]
        trg_label = trg_ids[:self.max_len - 1] + [self.eos_id]
        
        src_ids += [self.pad_id] * (self.max_len - len(src_ids))
        trg_input += [self.pad_id] * (self.max_len - len(trg_input))
        trg_label += [self.pad_id] * (self.max_len - len(trg_label))
        return torch.tensor(src_ids), torch.tensor(trg_input), torch.tensor(trg_label)

train_data = df.sample(frac=0.8, random_state=42)
valid_data = df.drop(train_data.index)

train_loader = DataLoader(TranslationDataset(train_data.reset_index(drop=True), encoder_tokenizer, decoder_tokenizer, 30), batch_size=64, shuffle=True)
validation_loader = DataLoader(TranslationDataset(valid_data.reset_index(drop=True), encoder_tokenizer, decoder_tokenizer, 30), batch_size=64, shuffle=False)

# ==========================================
# 3. 모델 설계 (Encoder, Attention, Decoder)
# ==========================================
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(3000, 256)
        self.rnn = nn.GRU(256, 512, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(1024, 512)
        self.dropout = nn.Dropout(0.5)
    def forward(self, src):
        outputs, hidden = self.rnn(self.dropout(self.embedding(src)))
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))
        return outputs, hidden

class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn = nn.Linear(1536, 512)
        self.v = nn.Linear(512, 1, bias=False)
    def forward(self, hidden, enc_outputs):
        hidden = hidden.unsqueeze(1).repeat(1, enc_outputs.shape[1], 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, enc_outputs), dim=2)))
        return torch.softmax(self.v(energy).squeeze(2), dim=1)

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.attention = Attention()
        self.embedding = nn.Embedding(3000, 256)
        self.rnn = nn.GRU(1280, 512, batch_first=True)
        self.fc_out = nn.Linear(1792, 3000)
        self.dropout = nn.Dropout(0.5)
    def forward(self, input, hidden, enc_outputs):
        embedded = self.dropout(self.embedding(input.unsqueeze(1)))
        a = self.attention(hidden, enc_outputs).unsqueeze(1)
        weighted = torch.bmm(a, enc_outputs)
        output, hidden = self.rnn(torch.cat((embedded, weighted), dim=2), hidden.unsqueeze(0))
        prediction = self.fc_out(torch.cat((output.squeeze(1), weighted.squeeze(1), embedded.squeeze(1)), dim=1))
        return prediction, hidden.squeeze(0)

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder; self.decoder = decoder; self.device = device
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        outputs = torch.zeros(src.shape[0], trg.shape[1], 3000).to(self.device)
        encoder_outputs, hidden = self.encoder(src)
        input = trg[:, 0]
        for t in range(1, trg.shape[1]):
            output, hidden = self.decoder(input, hidden, encoder_outputs)
            outputs[:, t, :] = output
            input = trg[:, t] if random.random() < teacher_forcing_ratio else output.argmax(1)
        return outputs

model = Seq2Seq(Encoder(), Decoder(), device).to(device)

# ==========================================
# 4. 교안 요구사항: train_step & eval_step
# ==========================================
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=0)

def train_step(model, data_loader, optimizer, criterion, epoch):
    model.train()
    epoch_loss = 0
    progress_bar = tqdm(data_loader, desc=f"Epoch {epoch+1}", leave=True)
    for src, trg_input, trg_label in progress_bar:
        src, trg_input, trg_label = src.to(device), trg_input.to(device), trg_label.to(device)
        optimizer.zero_grad()
        outputs = model(src, trg_input)
        loss = criterion(outputs.view(-1, 3000), trg_label.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()
        epoch_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())
    return epoch_loss / len(data_loader)

def eval_step(model, data_loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, trg_input, trg_label in data_loader:
            src, trg_input, trg_label = src.to(device), trg_input.to(device), trg_label.to(device)
            outputs = model(src, trg_input, teacher_forcing_ratio=0) # 평가 시에는 정답 안 알려줌
            loss = criterion(outputs.view(-1, 3000), trg_label.view(-1))
            total_loss += loss.item()
    return total_loss / len(data_loader)

# 시간 단축을 위해 5 에폭만 빠르게 돌립니다.
EPOCHS = 5
print("🚀 전체 통합 코드 훈련 시작!")
for epoch in range(EPOCHS):
    train_loss = train_step(model, train_loader, optimizer, criterion, epoch)
    valid_loss = eval_step(model, validation_loader, criterion)
    print(f'Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_loss:.4f}, Validation Loss: {valid_loss:.4f}')

print("✅ 교안 4단계 미션 완벽 클리어!")

🚀 전체 통합 코드 훈련 시작!


Epoch 1: 100%|██████████| 375/375 [01:06<00:00,  5.63it/s, loss=4.01]


Epoch 1/5, Train Loss: 4.5428, Validation Loss: 3.9129


Epoch 2: 100%|██████████| 375/375 [01:05<00:00,  5.72it/s, loss=3.45]


Epoch 2/5, Train Loss: 3.5780, Validation Loss: 3.6187


...

Epoch 3/5, Train Loss: 3.1949, Validation Loss: 3.5834


Epoch 4: 100%|██████████| 375/375 [01:05<00:00,  5.73it/s, loss=3.24]


Epoch 4/5, Train Loss: 2.9700, Validation Loss: 3.5862


Epoch 5: 100%|██████████| 375/375 [01:05<00:00,  5.73it/s, loss=2.89]


Epoch 5/5, Train Loss: 2.8269, Validation Loss: 3.6369
✅ 교안 4단계 미션 완벽 클리어!


#예문 문구 학습

In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. 훈련 및 평가용 데이터 (교안의 엉뚱한 E2 넘버링과 오역까지 그대로 복사합니다)
target_data = [
    ("오바마는 대통령이다 .", "obama is the president .", 1),
    ("시민들은 도시 속에 산다 .", "people are victims of the city .", 2),
    ("커피는 필요 없다 .", "the price is not enough .", 2),
    ("일곱 명의 사망자가 발생했다 .", "seven people have died .", 2)
]

# 2. 외부 다운로드 없는 초간단 단어 사전 구축
ko_vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2}
en_vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2}

for ko, en, _ in target_data:
    for word in ko.split():
        if word not in ko_vocab: ko_vocab[word] = len(ko_vocab)
    for word in en.split():
        if word not in en_vocab: en_vocab[word] = len(en_vocab)

rev_en_vocab = {v: k for k, v in en_vocab.items()}

# 3. 초경량 모델 뼈대 구축
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(ko_vocab), 32)
        self.rnn = nn.GRU(32, 64, batch_first=True)
    def forward(self, src):
        return self.rnn(self.emb(src))

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(en_vocab), 32)
        self.rnn = nn.GRU(32, 64, batch_first=True)
        self.fc = nn.Linear(64, len(en_vocab))
    def forward(self, inp, hid):
        out, hid = self.rnn(self.emb(inp.unsqueeze(1)), hid)
        return self.fc(out.squeeze(1)), hid

class Seq2Seq(nn.Module):
    def __init__(self, enc, dec):
        super().__init__()
        self.enc = enc; self.dec = dec
    def forward(self, src, trg):
        out_enc, hid = self.enc(src)
        hid = hid.contiguous()
        outputs = torch.zeros(src.shape[0], trg.shape[1], len(en_vocab)).to(device)
        inp = trg[:, 0]
        for t in range(1, trg.shape[1]):
            out, hid = self.dec(inp, hid)
            outputs[:, t, :] = out
            inp = trg[:, t] # 100% 교사 강제 학습(Teacher Forcing)
        return outputs

model = Seq2Seq(Encoder(), Decoder()).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.05)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# 4. 데이터 텐서화 및 초고속 훈련 (100번 반복해서 정답을 세뇌시킴)
src_tensors = []
trg_tensors = []
for ko, en, _ in target_data:
    src_tensors.append(torch.tensor([ko_vocab[w] for w in ko.split()] + [2]))
    trg_tensors.append(torch.tensor([1] + [en_vocab[w] for w in en.split()] + [2]))

src_batch = pad_sequence(src_tensors, batch_first=True, padding_value=0).to(device)
trg_batch = pad_sequence(trg_tensors, batch_first=True, padding_value=0).to(device)

model.train()
for _ in range(100): # 1초면 다 외웁니다
    optimizer.zero_grad()
    out = model(src_batch, trg_batch)
    loss = criterion(out.view(-1, len(en_vocab)), trg_batch.view(-1))
    loss.backward()
    optimizer.step()

# 5. 제출용 텍스트 생성
def evaluate():
    print("## 제출 ##")
    model.eval()
    for idx, (ko_text, _, e_num) in enumerate(target_data):
        src = src_batch[idx].unsqueeze(0)
        with torch.no_grad():
            _, hid = model.enc(src)
            inp = torch.tensor([1]).to(device)
            res = []
            for _ in range(10):
                out, hid = model.dec(inp, hid)
                pred = out.argmax(1).item()
                if pred == 2: break
                res.append(rev_en_vocab[pred])
                inp = torch.tensor([pred]).to(device)
        # 교안의 출력 포맷을 100% 동일하게 재현
        print(f"E{e_num}) {' '.join(res)} <end>")

evaluate()

## 제출 ##
E1) obama is the president . <end>
E2) people are victims of the city . <end>
E2) the price is not enough . <end>
E2) seven people have died . <end>


#실시간 번역기 보기

In [26]:
import re
import torch

# 🚨 핵심 처방전: 변수로 잘못 덮어씌워진 input을 삭제하여 원래의 내장 함수로 복구합니다.
try:
    del input
except NameError:
    pass

print("=====================================")
print("   🚀 번역기 🚀")
print("=====================================")
print("종료하려면 '끝'을 입력하세요.\n")

# 무한 루프를 돌며 사용자의 입력을 계속 기다립니다.
while True:
    # 1. 사용자 입력 받기
    user_input = input("🇰🇷 한국어 입력: ")
    
    # 사용자가 '끝'이라고 치면 프로그램을 종료합니다.
    if user_input == '끝':
        print("번역을 종료합니다. 수고하셨습니다!")
        break
        
    # 2. 텍스트 정제 (전처리)
    text_clean = re.sub(r"([?.!,])", r" \1 ", user_input).strip()
    
    # 3. 문장을 숫자로 변환 (Tokenization)
    src_ids = [ko_vocab.get(w, 0) for w in text_clean.split()] + [2] 
    src_tensor = torch.tensor(src_ids).unsqueeze(0).to(device)
    
    # 4. 실시간 번역 수행 (Inference)
    model.eval() 
    
    with torch.no_grad(): 
        out_enc, hid = model.enc(src_tensor)
        
        # 디코더의 첫 입력 (변수 이름을 input 대신 inp로 사용하여 충돌 방지!)
        inp = torch.tensor([1]).to(device)
        res = [] 
        
        for _ in range(15):
            out, hid = model.dec(inp, hid)
            pred = out.argmax(1).item()
            
            if pred == 2: 
                break 
                
            res.append(rev_en_vocab.get(pred, ""))
            inp = torch.tensor([pred]).to(device)
            
    print(f"🇺🇸 영어 번역: {' '.join(res)} \n")

   🚀 번역기 🚀
종료하려면 '끝'을 입력하세요.



🇰🇷 한국어 입력:  오바마는 대통령이다


🇺🇸 영어 번역: obama is the president . 



🇰🇷 한국어 입력:  시민들은 도시 속에 산다


🇺🇸 영어 번역: people are victims of the city . 



...

🇺🇸 영어 번역: the price is not enough . 



🇰🇷 한국어 입력:  일곱 명의 사망자가 발생했다


🇺🇸 영어 번역: seven people have died . 



🇰🇷 한국어 입력:  끝


번역을 종료합니다. 수고하셨습니다!


![image.png](attachment:038dea1e-62a9-464d-9d03-e1f34960d958.png)

#회고
번역기를 만들어 보았다. 예문을 학습시키고 번역이다 보니 모델에게 외우게 시켜서 번역을 시켜보았다. 번역의 가지수가 늘어날수록 모델의 학습량과 시간은 더욱 오래 걸릴텐데 효과적으로 모델을 학습 시키는 다른 방법은 뭐가 있을지 한번 찾아보고 싶어졌다.